# Laboratório 08: MLP com Keras

**Fundamentos de Inteligência Artificial** · 2026.2 · Prof. Leonardo Garcia Marques

---

Na aula passada (lab07) você escreveu um MLP **do zero**, em NumPy: pesos, forward
pass, perda, retropropagação e o laço de treino. Hoje as mesmas peças vêm prontas no
**Keras**, e a rede cresce: vamos classificar **fotos de roupas** em 10 classes, ver a
rede **sobreajustar** e segurar o sobreajuste com **dropout** e **parada antecipada**.

### Onde rodar

**No Google Colab (recomendado).** Baixe este notebook do AVA, abra
`colab.research.google.com`, entre com a sua conta Google e use o menu **Arquivo**
(*File*), opção de enviar notebook (*Upload notebook*). Não precisa de GPU: o lab
roda em poucos minutos na CPU do Colab. Se quiser tentar uma GPU (opcional), é no menu
**Ambiente de execução** (*Runtime*), alterar o tipo de ambiente (*Change runtime
type*); no Colab gratuito a GPU nem sempre está disponível.

**Na sua máquina (mais devagar).** Na pasta do projeto da disciplina, rode
`uv run --with tensorflow jupyter lab`. Na primeira vez o uv baixa o TensorFlow, que é
grande: demora.

Em qualquer dos dois, a Parte 2 baixa o conjunto de dados da internet na primeira
vez. Nada mais é baixado e o notebook não depende de arquivo seu.

### O que vamos fazer

| Parte | O que |
|---|---|
| 1 | a ponte: o XOR do lab07, agora em Keras |
| 2 | os dados: Fashion-MNIST |
| 3 | montar a rede e contar os parâmetros |
| 4 | o chute antes de treinar |
| 5 | treinar e ler as curvas de aprendizado |
| 6 | avaliar no teste e prever |
| 7 | a mesma rede com dropout e parada antecipada |
| 8 | três otimizadores |
| 9 | perguntas para você |

As lacunas estão marcadas com `# TODO`. Cada uma tem uma dica e um valor provisório,
para o notebook rodar inteiro mesmo antes de você preencher.

### Sobre os números (esperado: ...)

As **contas exatas** (parâmetros, passos por época) dão sempre o mesmo número. Já os
números de **treino** de rede neural **variam um pouco** com a versão do TensorFlow, a
máquina e o sorteio, mesmo com a semente fixa. Para esses, o esperado é uma **faixa**
(por exemplo, "entre 0.80 e 0.86"). Se o seu número cair fora dela, anote e mostre ao
professor: pode ser só a versão.

In [ ]:
import math
import time
import numpy as np
import matplotlib.pyplot as plt

try:
    import keras
    import tensorflow as tf
except ImportError:
    print("TensorFlow/Keras não encontrado. No Colab, avise o professor. Na sua máquina,")
    print("feche o Jupyter e abra de novo com:  uv run --with tensorflow jupyter lab")
    raise

print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
SEMENTE = 20261111

## Parte 1: a ponte, o XOR do lab07 em Keras (com 4 neurônios ocultos e tanh, para convergir em 300 épocas)

As seis peças do lab07 viram argumentos: `Dense` cria e sorteia os pesos e vieses,
`activation=` é a ativação, `compile(loss=...)` é a perda, `compile(optimizer=...)` é o
passo do gradiente, e o `fit` é o laço (ida, perda, volta, passo). No lab07 a rede era
2-2-1 com sigmoide (9 parâmetros) e precisava de milhares de épocas; aqui, com 4
neurônios e tanh, 300 bastam.
`batch_size=4`: os quatro exemplos entram em cada passo.

In [ ]:
keras.utils.set_random_seed(SEMENTE)
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype="float32")
y_xor = np.array([0, 1, 1, 0], dtype="float32")

xor = keras.Sequential([
    keras.Input(shape=(2,)),                      # 2 entradas
    keras.layers.Dense(4, activation="tanh"),     # camada oculta: 4 neurônios
    keras.layers.Dense(1, activation="sigmoid"),  # saída: 1 neurônio
])
xor.compile(loss="mse", optimizer=keras.optimizers.SGD(learning_rate=1.0))

inicio = time.time()
xor.fit(X_xor, y_xor, epochs=300, batch_size=4, verbose=0)
print(f"treino em {time.time() - inicio:.0f} s")

saida = xor.predict(X_xor, verbose=0).ravel()
print("saídas da rede:", saida.round(2))
print("arredondado:", saida.round().astype(int), "(esperado: [0 1 1 0])")

In [ ]:
# TODO 1: quantos parâmetros tem esse MLP? Conte À MÃO (entradas x neurônios + vieses, por camada).
# Dica: a camada oculta recebe 2 entradas e tem 4 neurônios; a de saída recebe 4 e tem 1.
xor_a_mao = 0   # valor provisório

print("Keras:", xor.count_params(), "(esperado: 17)")
print("à mão:", xor_a_mao, "(esperado: 17)")

## Parte 2: os dados, Fashion-MNIST

60.000 fotos de treino e 10.000 de teste, $28 \times 28$ pixels em tons de cinza, 10
classes de roupas (documentação do Keras, `fashion_mnist`). É o conjunto do capítulo 10
da 3ª edição do Géron. A primeira execução desta célula baixa os dados.

In [ ]:
(X_treino_todo, y_treino_todo), (X_teste, y_teste) = keras.datasets.fashion_mnist.load_data()

NOMES = ["camiseta", "calça", "pulôver", "vestido", "casaco",
         "sandália", "camisa", "tênis", "bolsa", "bota"]

print("formato do treino:", X_treino_todo.shape, "(esperado: (60000, 28, 28))")
print("formato do teste:", X_teste.shape, "(esperado: (10000, 28, 28))")
print("maior pixel:", X_treino_todo.max(), "(esperado: 255)")

fig, eixos = plt.subplots(2, 10, figsize=(12, 3))
for i, ax in enumerate(eixos.ravel()):
    ax.imshow(X_treino_todo[i], cmap="binary")
    ax.set_title(NOMES[y_treino_todo[i]], fontsize=9)
    ax.axis("off")
plt.show()

In [ ]:
# TODO 2: leve os pixels de 0..255 para 0..1, no treino E no teste.
# Dica: divida por 255.0 (o notebook oficial do Géron faz X_train / 255.).
X_treino_todo = X_treino_todo * 1.0   # valor provisório: NÃO dividiu
X_teste = X_teste * 1.0               # valor provisório: NÃO dividiu

print("maior pixel agora:", X_treino_todo.max(), "(esperado: 1.0)")

Para caber numa CPU, treinamos com as **5000 primeiras** imagens. As
**5000 últimas** do conjunto de treino ficam para **validação**, como no notebook
oficial (que treina com 55.000). O teste continua com as 10.000.

In [ ]:
X_treino, y_treino = X_treino_todo[:5000], y_treino_todo[:5000]
X_valid, y_valid = X_treino_todo[-5000:], y_treino_todo[-5000:]

print("treino:", len(X_treino), "(esperado: 5000)")
print("validação:", len(X_valid), "(esperado: 5000)")

## Parte 3: montar a rede e contar os parâmetros

A rede do livro: `Flatten` enfileira os 784 pixels; duas camadas ocultas (300 e 100
neurônios, ReLU); a saída com 10 neurônios e **softmax**, uma probabilidade por classe.
A função abaixo monta a rede; o parâmetro `taxa_dropout` só será usado na Parte 7.

In [ ]:
def montar_rede(taxa_dropout=0.0):
    camadas = [keras.Input(shape=(28, 28)), keras.layers.Flatten()]
    for neuronios in (300, 100):
        camadas.append(keras.layers.Dense(neuronios, activation="relu"))
        if taxa_dropout > 0:
            camadas.append(keras.layers.Dropout(taxa_dropout))
    camadas.append(keras.layers.Dense(10, activation="softmax"))
    return keras.Sequential(camadas)

keras.utils.set_random_seed(SEMENTE)
rede = montar_rede()
rede.summary()

In [ ]:
# TODO 3: conte À MÃO os parâmetros da rede (entradas x neurônios + vieses, camada por camada).
# Dica: a 1ª Dense recebe 784 entradas; a 2ª recebe a SAÍDA da 1ª (300), não 784.
rede_a_mao = 0   # valor provisório

print("Keras:", rede.count_params(), "(esperado: 266610)")
print("à mão:", rede_a_mao, "(esperado: 266610)")

**Pergunte-se:** quantos desses parâmetros estão na primeira camada? Por que o
`Flatten` tem 0?

## Parte 4: o chute antes de treinar

A perda é a **entropia cruzada**: para cada imagem, $-\ln$(probabilidade que a rede deu
à classe certa). Como os rótulos são inteiros (0 a 9), usamos
`sparse_categorical_crossentropy`. Uma rede que ainda não aprendeu nada dá mais ou menos
0,1 a cada classe, então a perda inicial deve ficar perto de $-\ln 0{,}1$.

In [ ]:
# TODO 4: calcule a perda de um chute uniforme em 10 classes: -ln(1/10).
# Dica: np.log é o logaritmo natural.
chute = 0.0   # valor provisório

print(f"perda do chute: {chute:.4f} (esperado: 2.3026)")

In [ ]:
rede.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
perda0, acc0 = rede.evaluate(X_valid, y_valid, verbose=0)
print(f"perda antes de treinar: {perda0:.2f} (esperado: entre 2.1 e 2.8)")
print(f"acurácia antes de treinar: {acc0:.2f} (esperado: entre 0.0 e 0.2)")

Se a perda antes de treinar saiu muito acima de 2,3, confira a Parte 2: pixels de
0 a 255 (sem dividir) dão números enormes na entrada.

## Parte 5: treinar e ler as curvas de aprendizado

30 épocas com o otimizador Adam e lote de 32 (o padrão do Keras). O `fit` mede a
validação no fim de cada época e devolve o **histórico**. Numa CPU leva perto de um
minuto (no container de preparo, de 35 a 65 s).

In [ ]:
inicio = time.time()
hist = rede.fit(X_treino, y_treino, epochs=30, validation_data=(X_valid, y_valid), verbose=0)
print(f"treino em {time.time() - inicio:.0f} s")
H = hist.history
print("chaves do histórico:", list(H.keys()))

In [ ]:
# TODO 5: quantos passos (atualizações dos pesos) tem cada época?
# Dica: exemplos de treino / lote, arredondado PARA CIMA (math.ceil).
passos_a_mao = 0   # valor provisório

print("passos por época (Keras):", hist.params["steps"], "(esperado: 157)")
print("passos por época (à mão):", passos_a_mao, "(esperado: 157)")

In [ ]:
epocas = range(1, len(H["loss"]) + 1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(epocas, H["loss"], "o-", label="treino")
ax[0].plot(epocas, H["val_loss"], "o-", label="validação")
ax[0].set_title("perda"); ax[0].set_xlabel("época"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(epocas, H["accuracy"], "o-", label="treino")
ax[1].plot(epocas, H["val_accuracy"], "o-", label="validação")
ax[1].set_title("acurácia"); ax[1].set_xlabel("época"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.show()

In [ ]:
# TODO 6: em que época a perda de VALIDAÇÃO foi a menor? (conte as épocas a partir de 1)
# Dica: np.argmin(H["val_loss"]) dá a posição, contando de 0.
melhor_epoca = 0   # valor provisório

print("melhor época:", melhor_epoca, "(esperado: entre 2 e 15)")
print("a perda de validação final é maior que a mínima:", H["val_loss"][-1] > min(H["val_loss"]), "(esperado: True)")
print(f"acurácia de treino na última época: {H['accuracy'][-1]:.2f} (esperado: entre 0.9 e 0.99)")
print(f"acurácia de validação na última época: {H['val_accuracy'][-1]:.2f} (esperado: entre 0.8 e 0.87)")

**Leia o gráfico:** a perda de treino cai quase o tempo todo (um solavanco no fim não
é sobreajuste: é o otimizador, com taxa fixa, oscilando perto do mínimo). A de validação
cai nas primeiras épocas e depois **sobe**: é o **sobreajuste**. A rede tem 266.610 parâmetros para 5.000
imagens e começa a decorar o treino. A sua melhor época pode ser diferente da do
colega: o formato da curva é o que se repete.

## Parte 6: avaliar no teste e prever

O teste fica para o fim. `evaluate` devolve a perda e a acurácia; `predict` devolve,
para cada imagem, as 10 probabilidades da softmax.

In [ ]:
perda_teste, acc_teste = rede.evaluate(X_teste, y_teste, verbose=0)
print(f"perda no teste: {perda_teste:.2f} (esperado: entre 0.6 e 1.0)")
print(f"acurácia no teste: {acc_teste:.2f} (esperado: entre 0.8 e 0.86)")

In [ ]:
proba = rede.predict(X_teste[:3], verbose=0)
print("formato:", proba.shape, "(esperado: (3, 10))")
print(f"soma das probabilidades da 1ª imagem: {proba[0].sum():.3f} (esperado: entre 0.999 e 1.001)")

# TODO 7: qual a classe prevista para cada uma das 3 imagens? (a posição da maior probabilidade)
# Dica: proba.argmax(axis=1)
previstas = np.zeros(3, dtype=int)   # valor provisório

print("classes verdadeiras:", y_teste[:3], "(esperado: [9 2 1])")
print("classes previstas:", previstas)
print("nomes previstos:", [NOMES[c] for c in previstas])

Compare as previstas com as verdadeiras (9 = bota, 2 = pulôver, 1 = calça). A rede
acerta a maioria das imagens fáceis; os erros costumam ficar entre classes parecidas
(camisa, camiseta, pulôver, casaco).

## Parte 7: a mesma rede com dropout e parada antecipada

- **Dropout**: a cada passo de treino, cada neurônio da camada é desligado com
  probabilidade `taxa_dropout`; na previsão, ninguém é desligado.
- **Parada antecipada** (`EarlyStopping`): olha a perda de validação (`val_loss`) a cada
  época; se ela passar `patience` épocas seguidas sem melhorar, para o treino; com
  `restore_best_weights=True`, volta aos pesos da melhor época.

In [ ]:
# TODO 8: use dropout de 0.3 depois de cada camada oculta.
TAXA_DROPOUT = 0.0   # valor provisório: sem dropout

# TODO 9: crie a parada antecipada: paciência de 5 épocas, voltando aos melhores pesos.
# Dica: keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
#       e depois  callbacks = [parada]
callbacks = []   # valor provisório: sem parada antecipada

keras.utils.set_random_seed(SEMENTE)
rede_reg = montar_rede(TAXA_DROPOUT)
rede_reg.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
inicio = time.time()
hist_reg = rede_reg.fit(X_treino, y_treino, epochs=30, validation_data=(X_valid, y_valid),
                        verbose=0, callbacks=callbacks)
print(f"treino em {time.time() - inicio:.0f} s")
HR = hist_reg.history

In [ ]:
epocas_rodadas = len(HR["loss"])
melhor_reg = int(np.argmin(HR["val_loss"])) + 1
print("épocas rodadas:", epocas_rodadas, "  melhor época:", melhor_reg)
print("parou 5 épocas depois da melhor (ou foi até 30):",
      epocas_rodadas == min(melhor_reg + 5, 30), "(esperado: True)")
print("dropout não tem parâmetro: mesma contagem:", rede_reg.count_params() == rede.count_params(), "(esperado: True)")

perda_teste_reg, acc_teste_reg = rede_reg.evaluate(X_teste, y_teste, verbose=0)
print(f"perda no teste, com regularização: {perda_teste_reg:.2f} (esperado: entre 0.38 e 0.6)")
print(f"acurácia no teste, com regularização: {acc_teste_reg:.2f} (esperado: entre 0.8 e 0.87)")
print("a perda de teste caiu com a regularização:", perda_teste_reg < perda_teste, "(esperado: True)")

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(H["val_loss"]) + 1), H["val_loss"], "o-", label="sem regularização")
plt.plot(range(1, epocas_rodadas + 1), HR["val_loss"], "s-", label="dropout + parada antecipada")
plt.xlabel("época"); plt.ylabel("perda de validação"); plt.legend(); plt.grid(alpha=.3)
plt.show()

**Leia com honestidade:** a perda de teste costuma cair bastante; a acurácia, às vezes
só um pouco. A rede regularizada erra menos "convencida" (dá menos probabilidade quase
zero à classe certa), e ninguém precisou adivinhar o número de épocas.

Compare também o ponto mais baixo das duas curvas do gráfico: se forem parecidos, o ganho
na perda de teste veio da **parada antecipada**, não do dropout. Para isolar o dropout,
rode esta Parte de novo com `TAXA_DROPOUT = 0.0` e a parada ligada.

Repare também: com dropout, a acurácia de **treino** que o `fit` mostra pode ficar
**abaixo** da de validação. Não é erro: no treino a rede está "desfalcada" (o notebook
oficial do cap. 11 comenta isso).

## Parte 8: três otimizadores

A mesma rede, o mesmo sorteio inicial, 10 épocas, três otimizadores: o SGD comum (taxa
0,01, o padrão do Keras), o SGD com **momento** 0,9 e o **Adam** (taxa 0,001, o padrão).
Numa CPU leva perto de um minuto.

In [ ]:
# TODO 10: crie o SGD com momento 0.9 (mesma taxa, 0.01).
# Dica: keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)
def sgd_com_momento():
    return keras.optimizers.SGD(learning_rate=0.01, momentum=0.0)   # valor provisório: sem momento

otimizadores = {
    "SGD": lambda: keras.optimizers.SGD(learning_rate=0.01),
    "SGD com momento": sgd_com_momento,
    "Adam": lambda: keras.optimizers.Adam(learning_rate=0.001),
}

In [ ]:
historicos = {}
inicio = time.time()
for nome, criar in otimizadores.items():
    keras.utils.set_random_seed(SEMENTE)
    r = montar_rede()
    r.compile(loss="sparse_categorical_crossentropy", optimizer=criar(), metrics=["accuracy"])
    historicos[nome] = r.fit(X_treino, y_treino, epochs=10, validation_data=(X_valid, y_valid),
                             verbose=0).history
print(f"treino em {time.time() - inicio:.0f} s")
for nome, Ho in historicos.items():
    print(f"{nome:<16} perda de treino {Ho['loss'][-1]:.3f}   acurácia de validação {Ho['val_accuracy'][-1]:.3f}")

print("o momento baixou a perda de treino do SGD:",
      historicos["SGD com momento"]["loss"][-1] < historicos["SGD"]["loss"][-1], "(esperado: True)")
print("o Adam baixou a perda de treino do SGD:",
      historicos["Adam"]["loss"][-1] < historicos["SGD"]["loss"][-1], "(esperado: True)")

plt.figure(figsize=(8, 4))
for nome, Ho in historicos.items():
    plt.plot(range(1, 11), Ho["loss"], "o-", label=nome)
plt.xlabel("época"); plt.ylabel("perda de treino"); plt.legend(); plt.grid(alpha=.3)
plt.show()

Em 10 épocas, o SGD comum ainda está descendo devagar; o momento e o Adam descem
mais rápido. Não é uma lei: com mais épocas ou outra taxa, a ordem pode mudar.

## Parte 9: perguntas para você (não vale nota)

1. Na Parte 1, em que linha do código está a retropropagação que você escreveu no lab07?
2. Por que a segunda camada da Parte 3 tem 300 entradas, e não 784?
3. Por que a perda antes de treinar fica perto de 2,30? O que aconteceria sem dividir
   os pixels por 255?
4. Em 30 épocas, quantas vezes os pesos da rede da Parte 5 foram atualizados?
5. Olhando só a curva de treino da Parte 5, você perceberia o sobreajuste? Por quê?
6. O que a parada antecipada da Parte 7 teria feito com `restore_best_weights=False`?
7. Na Parte 8, por que as três redes começam com os mesmos pesos? Por que isso importa
   para comparar os otimizadores?